# Final Project - ST - 554 Big data analysis

## Author: Yefrid Cordoba

### Fitting model

In this section we are going to use the whole data from the file `'power_ml_data.csv'` for the fitting of the models, using first a principal component analysis to pick the most important features, which are going to fe fed to a elastic net model crossvalidation.

We import the modules that are going to be used

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, StandardScaler, OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

And create the spark session

In [2]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 17:44:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 17:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/27 17:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/27 17:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/27 17:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
spark.catalog.clearCache()

Now we import the data as a pandas dataframe, and then we are going to convert it to a spark dataframe.

In [4]:
power_fit = pd.read_csv('power_ml_data.csv')

In [5]:
#We convert to spark dataframe
power_fit = spark.createDataFrame(power_fit)

In [6]:
my_schema = power_fit.schema

In [7]:
power_fit.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We are going to transformthe hour column from a `LongType()` to a `DoubleType()`.

In [8]:
type_double = SQLTransformer(
    statement="""
        SELECT * EXCEPT (Hour), 
        CAST(Hour AS DOUBLE) AS Hour 
        FROM __THIS__
    """
)

Also, we want to binarize the Hour column based on it being less than 6.5 or not.

In [9]:
 bin_hour = SQLTransformer( 
    statement = """
            SELECT *,
            CASE WHEN Hour < 6.5 THEN 1 ELSE 0 END AS bin_Hour
            FROM __THIS__
            """
 )

In [10]:
bin_hour.transform(type_double.transform(power_fit)).show(20)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|bin_Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1| 0.0|       1|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1| 0.0|       1|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1| 0.0|       1|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1| 0.0|       1|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 1

Now we ar going to create a column with the `features`, and rename the response column as `label`.

In [11]:
feature_cols = ['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows']
pca_assem = VectorAssembler(inputCols = feature_cols,\
                         outputCol = 'feature_PC')

In [12]:
rename_col = SQLTransformer( 
    statement = """
            SELECT *, Power_Zone_3 as label
            FROM __THIS__
            """
    )

Also as PCA is sensitive to magnitude of the feautres, we are going to standarize the feature column, process that is going to be repeated when setting up the feautres for the elastic net regression.

In [13]:
scaler = StandardScaler(
    inputCol='feature_PC', 
    outputCol='scaled_features_PC',
    withMean=True,
    withStd=True
)

We use the OneHotEcoder for the month variable, setting droplast = True, so it recognizes that when all the codes are 0, then it is the last month.

In [14]:
month_encoder = OneHotEncoder(
    inputCol='Month',
    outputCol="Month_Encoded",
    dropLast=True
)

In [15]:
month_encoder.fit(power_fit).transform(power_fit).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour| Month_Encoded|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|(12,[1],[1.0])|
|      5.921|    75.7|     0.081|                0.048|

With the standarized features, we set up the stimator for the PCA, using 2 principal components.

In [16]:
pca = PCA(
    k=2,
    inputCol='scaled_features_PC', 
    outputCol='pca_features'
)

Now we create the transformer for the standard scaling of the missing continuous variables that were not scaled for the PCA.

In [17]:
feature_p = ['Power_Zone_1', 'Power_Zone_2']
pz_assem = VectorAssembler(inputCols = feature_p,\
                         outputCol = 'feature_pz')

In [18]:
scaler_pz = StandardScaler(
    inputCol='feature_pz', 
    outputCol='scaled_features_pz',
    withMean=True,
    withStd=True
)

And finally we can put all the features(from PCA, scaled and binary) in one column that is going to be used when fitting the Elastic net.

In [19]:
feature_assem = VectorAssembler(inputCols = ['pca_features', 'scaled_features_pz', 'bin_Hour', 'Month_Encoded'],\
                         outputCol = 'features')

We can add linear regression as one of the transformations for our pipeline, which later is going to be used for the cross validator.

In [20]:
lr = LinearRegression()

Now that we have all the transformations we can fit our pipeline.

In [21]:
pipeline = Pipeline(stages = [type_double, bin_hour, pca_assem, rename_col, scaler, month_encoder, pca, pz_assem, scaler_pz, feature_assem, lr])

In [22]:

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5,
                          parallelism = 128)

In [23]:
cvModel = crossval.fit(power_fit)

26/04/27 17:44:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 17:45:00 WARN Instrumentation: [f5f81f29] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 17:45:00 WARN Instrumentation: [0e77e5bd] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 17:45:00 WARN Instrumentation: [87a4d032] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 17:45:01 WARN Instrumentation: [dfb35500] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 17:45:01 WARN Instrumentation: [f15af394] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 17:45:01 WARN Instrumentation: [3786696a] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 17:45:01 WARN Instrumentation: [71a7288a] regP

Now we can get the best regression parameter.

In [31]:
cvModel.bestModel.stages[-1].getRegParam()

0.9

And the best Elastic net parameter from the cross validation process.

In [32]:
cvModel.bestModel.stages[-1].getElasticNetParam()

0.1

Also, we report the cross-validation error, which is the lowest RMSE value for the combination (optimal parameters).

In [26]:
min(cvModel.avgMetrics)

2175.467527678137

Now we report the RMSE for the model when fitted to the whole dataset that was used in the cross validation process.

In [27]:
RegressionEvaluator().evaluate(cvModel.transform(power_fit))

2174.4520820940966

From this we can see that the RMSE for the whole dataset is sligthly smaller than it was for the cross validation for the optimal parameters.

We are going to delete all the temp columns from previous transformations.

In [28]:
pred_power = cvModel.transform(power_fit)

And calculate the residals for the fitted model.

In [29]:
res_power = pred_power.withColumn('residuals', pred_power.label - pred_power.prediction)\
                      .select('label', 'prediction', 'residuals')

In [30]:
res_power.show(5)

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386|20936.249778909325|-695.2859189093251|
|20131.08434| 18698.52452469986| 1432.559815300141|
|19668.43373|18234.142955809097| 1434.290774190904|
|18899.27711|17612.938962257354|1286.3381477426447|
|18442.40964|17014.420496912477| 1427.989143087525|
+-----------+------------------+------------------+
only showing top 5 rows


### Streaming

As the schema was already stored in an object called `my_schema`, we can use that as for the reading data step, including that the header is included.

In [33]:
new_power = spark.readStream.schema(my_schema).csv("power_data", header = True)

We write what we want to do with the data once is imported.

In [34]:
streamed_power = cvModel.transform(new_power)

We create the residuals columns, and the select just the columns `label`, `prediction`, and `residuals`.

In [35]:
pred_val = streamed_power\
               .withColumn('residuals', streamed_power.label - streamed_power.prediction)\
               .select('label', 'prediction', 'residuals')

In the same stream we are going to rename the output column to do the join on the same column that is being renamed in the transform method of the cvModel.

In [36]:
raw_power = new_power.withColumnRenamed('Power_Zone_3', 'label')

And we are going to start the query for joinig the tables.

In [37]:
joinquery = pred_val \
                .join(raw_power, "label", "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

26/04/27 17:52:24 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-015c7cc3-d8f7-484f-a868-98f6553a6795. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 17:52:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14460.25105|21128.193676544633|-6667.942626544633|      24.01|    83.0|     4.915|                0.424|        0.389| 22306.44518| 15026.58228|    7|   5|
|    18176.0|17788.442770216956|  387.557229783044|      20.02|   51.92|     0.083|                404.0|        383.4| 32972.74489| 18593.89002|    4|  17|
|18020.40486| 17595.73449541698| 424.6703645830203|      21.85|   58.09|     0.071|                843.0|        52.94

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10259.27711| 9952.809684684626|  306.4674253153753|       11.4|    80.3|     4.919|                297.2|        45.16| 25624.61538| 17959.09091|   11|  10|
|7341.176471| 7382.970156756201| -41.79368575620083|      11.67|    79.3|     0.086|                 88.4|         22.0| 24693.53612| 18833.99816|   12|   9|
|11178.79518| 9641.771197017128| 1537.0239829828715|      4.509|    74.5|     0.084|                6.643|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25891.08434|24907.511226901937|  983.5731130980639|      14.99|    74.8|     0.071|                 0.07|        0.115| 42027.34177| 25469.90881|    1|  19|
|14869.78723|13332.072013546729| 1537.7152164532708|      18.61|    84.2|     0.085|                0.069|        0.089| 33973.91685| 20404.97925|   10|  23|
|25266.95925| 26488.68510914878|-1221.7258591487807|      25.49|   55.92|      4.92|                708.0|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|27432.72727|  26653.6498668427|  779.0774031572982|       18.4|    73.0|     0.078|                3.077|        2.836| 43568.91281| 23352.34216|    4|  20|
|15398.70968|16482.941995593887|-1084.2323155938866|      20.71|   61.77|     4.916|                297.0|        322.1| 31796.42553| 19979.26829|    3|  17|
|9323.409364| 9436.775901873032|-113.36653787303294|       8.02|    85.3|     0.089|                286.2|       

...

-------------------------------------------
Batch: 12
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15004.94472|14462.712799483641|  542.2319205163585|      11.23|   62.45|     4.917|                0.059|        0.193| 23528.13559| 13925.83587|    2|   2|
|23536.24615| 24504.92702752278| -968.6808775227801|       20.6|    70.8|     0.079|                0.059|          0.1| 37573.50993| 23632.01663|    6|   1|
|11639.85594|10717.371948239688|  922.4839917603113|      12.33|    76.2|     0.078|                0.055|      

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12231.97568|13648.617737736085| -1416.642057736086|      20.66|    82.3|     4.915|                224.2|        171.3| 33885.68928| 25674.27386|   10|  13|
|11125.16129| 12746.84332154226|-1621.6820315422592|      13.36|    86.4|     0.083|                0.029|        0.078| 21986.04255| 13693.90244|    3|   6|
|    13920.0|13746.541488958508| 173.45851104149187|      13.55|    84.8|     0.075|                0.062|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12319.51368|14713.613264449832| -2394.099584449832|      23.36|   67.11|     4.921|                184.1|        158.0| 36097.68053| 23392.53112|   10|  14|
|12549.39759|14164.082981464657|-1614.6853914646563|      5.814|    83.1|     0.085|                13.12|        11.54| 26053.67089|  17576.8997|    1|   8|
|28683.63636|27600.810945415218| 1082.8254145847823|      18.17|    73.0|     0.079|                11.67|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13873.42186| 13512.40197161692|  361.0198883830799|      21.46|   56.55|     0.271|                0.095|        0.093|  29347.9646| 17259.04366|    9|   1|
|14385.52764|13560.236946004548|  825.2906939954519|      13.28|   66.66|     0.084|                0.051|        0.115|  22924.0678| 13706.99088|    2|   5|
|30553.30544|27659.041149847082|  2894.264290152918|      27.95|    71.8|     4.916|                308.0|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25216.64322| 22544.97104214321|2671.6721778567917|       8.39|   54.42|     0.085|                0.062|        0.108| 39459.66102| 22964.13374|    2|  20|
|  12436.231|14081.367168912517|-1645.136168912517|      20.76|    84.8|     0.067|                27.31|        23.58| 34881.40044|  22847.3029|   10|  15|
|14227.13085|14977.560440746602|-750.4295907466021|      17.47|   66.01|     0.085|                16.06|         18.

In [38]:
joinquery.stop()

26/04/27 17:56:10 WARN DAGScheduler: Failed to cancel job group d0fd1fdd-8609-4db4-9b5a-a881c2bac64e. Cannot find active jobs for it.
26/04/27 17:56:11 WARN DAGScheduler: Failed to cancel job group d0fd1fdd-8609-4db4-9b5a-a881c2bac64e. Cannot find active jobs for it.


Some of the outputs contain more rows, however, after checking the streamed data, all the samples contain 5 sample as requested.